In [1]:
import spacy
import numpy as np
import pandas as pd
from spacy.training import offsets_to_biluo_tags
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from keras.preprocessing.sequence import pad_sequences
from pytorch_pretrained_bert import BertTokenizer, BertConfig
from pytorch_pretrained_bert import BertForTokenClassification, BertModel
from seqeval.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import re
import logging
from tqdm import tqdm

In [2]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")
# Adding '\n' to the default spacy tokenizer
prefixes = ('\n', ) + tuple(nlp.Defaults.prefixes)
prefix_regex = spacy.util.compile_prefix_regex(prefixes)
nlp.tokenizer.prefix_search = prefix_regex.search

In [4]:
# Personal Custom Tags Dictionary
entity_dict = {
    'Name': 'NAME', 
    'College Name': 'CLG',
    'Degree': 'DEG',
    'Graduation Year': 'GRADYEAR',
    'Years of Experience': 'YOE',
    'Companies worked at': 'COMPANY',
    'Designation': 'DESIG',
    'Skills': 'SKILLS',
    'Location': 'LOC',
    'Email Address': 'EMAIL'
}
# Define tag2idx mapping (same as in training)
tag2idx = {
    'I-DESIG': 0, 'I-COMPANY': 1, 'B-COMPANY': 2, 'L-LOC': 3, 'U-DESIG': 4,
    'I-DEG': 5, 'L-SKILLS': 6, 'I-CLG': 7, 'U-DEG': 8, 'X': 9, 'I-LOC': 10,
    'L-DEG': 11, 'I-EMAIL': 12, 'O': 13, 'I-YOE': 14, 'B-NAME': 15, '[CLS]': 16,
    'L-GRADYEAR': 17, 'I-GRADYEAR': 18, 'U-SKILLS': 19, 'I-SKILLS': 20, 'L-DESIG': 21,
    'L-YOE': 22, 'U-YOE': 23, 'B-EMAIL': 24, 'L-NAME': 25, '[SEP]': 26, 'B-GRADYEAR': 27,
    'L-COMPANY': 28, 'L-EMAIL': 29, 'B-CLG': 30, 'I-NAME': 31, 'U-CLG': 32, 'L-CLG': 33,
    'U-GRADYEAR': 34, 'B-LOC': 35, 'U-EMAIL': 36, 'U-COMPANY': 37, 'B-DESIG': 38,
    'B-DEG': 39, 'U-LOC': 40, 'B-SKILLS': 41, 'B-YOE': 42
}
idx2tag = {tag2idx[key]: key for key in tag2idx.keys()}

In [5]:
# IMPROVEMENT 1: Add regex patterns for common resume entities to enhance detection
EMAIL_REGEX = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
PHONE_REGEX = r'(\+\d{1,3}[-.\s]?)?(\d{3}[-.\s]?\d{3}[-.\s]?\d{4}|$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4})'
YEAR_REGEX = r'\b(19|20)\d{2}\b'
LINKEDIN_REGEX = r'linkedin\.com\/in\/[a-zA-Z0-9_-]+'

In [6]:
# IMPROVEMENT 2: Enhanced text preprocessing
def preprocess_text(text):
    """Clean and normalize text for better entity recognition"""
    # Convert multiple spaces to single space
    text = re.sub(r'\s+', ' ', text)
    # Normalize bullet points and special characters
    text = re.sub(r'•|·|⋅|∙|⦁', '* ', text)
    # Ensure proper spacing after punctuation
    text = re.sub(r'([.,!?;:])\s*', r'\1 ', text)
    # Normalize newlines
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

In [7]:
# Function to merge overlapping intervals with improved logic
def mergeIntervals(intervals):
    if not intervals:
        return []
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []
    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            # Check for overlap
            if higher[0] <= lower[1]:
                # IMPROVEMENT 3: Better handling of overlapping entities
                # If same entity type, extend the range
                if lower[2] == higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                # If different entity types, prioritize based on entity importance
                else:
                    # Define priority order (you can customize this)
                    priority_order = {
                        'NAME': 1, 'EMAIL': 2, 'SKILLS': 3, 'COMPANY': 4, 
                        'DESIG': 5, 'DEG': 6, 'CLG': 7, 'GRADYEAR': 8, 
                        'YOE': 9, 'LOC': 10
                    }
                    # Choose entity with higher priority (lower number)
                    if priority_order.get(lower[2], 999) <= priority_order.get(higher[2], 999):
                        # Keep lower but adjust end if needed
                        if higher[1] > lower[1]:
                            merged[-1] = (lower[0], higher[1], lower[2])
                    else:
                        # Replace with higher
                        merged[-1] = higher
            else:
                merged.append(higher)
    return merged

In [8]:
# Extract entities from annotations with improved error handling
def get_entities(df):
    entities = []
    for i in range(len(df)):
        entity = []
        if df['annotation'][i] is not None:
            for annot in df['annotation'][i]:
                try:
                    # IMPROVEMENT 4: Better error handling for annotations
                    if 'label' in annot and len(annot['label']) > 0 and 'points' in annot:
                        label = annot['label'][0]
                        if label in entity_dict:
                            ent = entity_dict[label]
                            start = annot['points'][0]['start']
                            end = annot['points'][0]['end'] + 1
                            entity.append((start, end, ent))
                        else:
                            logger.warning(f"Unknown label: {label}")
                    else:
                        logger.warning(f"Malformed annotation: {annot}")
                except Exception as e:
                    logger.error(f"Error processing annotation: {e}")
                    continue
            # IMPROVEMENT 5: Apply entity merging only if entities exist
            if entity:
                entity = mergeIntervals(entity)
            entities.append(entity)
        else:
            entities.append([])
    return entities

In [9]:
# IMPROVEMENT 6: Enhanced data preparation with better sentence segmentation
def get_test_data(df):
    tags = []
    sentences = []
    for i in tqdm(range(len(df)), desc="Processing documents"):
        text = preprocess_text(df['content'][i])
        entities = df['entities'][i]
        # Use spaCy's sentence segmentation
        doc = nlp(text)
        # Get BILUO tags
        try:
            tag = offsets_to_biluo_tags(doc, entities)
        except Exception as e:
            logger.error(f"Error in BILUO tagging: {e}")
            # Fallback to 'O' tags if there's an error
            tag = ['O'] * len(doc)
        # Convert '-' tags to 'O'
        tag = ['O' if t == '-' else t for t in tag]
        # IMPROVEMENT 7: Better sentence segmentation
        # Split by sentences but keep entity context
        sents = list(doc.sents)
        if len(sents) == 0:  # Handle empty documents
            continue
        # Process each sentence
        for sent in sents:
            # Get the start and end indices of the sentence
            start_idx = sent[0].i
            end_idx = sent[-1].i + 1
            # Extract tokens and tags for this sentence
            sent_tokens = list(doc)[start_idx:end_idx]
            sent_tags = tag[start_idx:end_idx]
            # Only add sentences that have at least one non-O tag
            if any(t != 'O' for t in sent_tags):
                sentences.append(sent_tokens)
                tags.append(sent_tags)
    logger.info(f"Extracted {len(sentences)} sentences with entities")
    return sentences, tags

In [10]:
# IMPROVEMENT 8: Enhanced tokenization with better handling of special tokens
def get_tokenized_test_data(sentences, tags, tokenizer, max_len=256):
    tokenized_texts = []
    word_piece_labels = []
    attention_masks = []
    for word_list, label in tqdm(zip(sentences, tags), desc="Tokenizing", total=len(sentences)):
        # Initialize with special tokens
        tokens = ['[CLS]']
        labels = ['[CLS]']
        # Process each word
        for word, lab in zip(word_list, label):
            word_tokens = tokenizer.tokenize(word.text)
            # Handle empty tokens
            if not word_tokens:
                word_tokens = ['[UNK]']
            # Add tokens and corresponding labels
            for i, token in enumerate(word_tokens):
                tokens.append(token)
                # First token gets the actual label, rest get 'X'
                if i == 0:
                    labels.append(lab)
                else:
                    labels.append('X')
        # Add end token
        tokens.append('[SEP]')
        labels.append('[SEP]')
        # Truncate if too long
        if len(tokens) > max_len - 2:  # -2 for [CLS] and [SEP]
            tokens = tokens[:max_len-1]
            tokens.append('[SEP]')
            labels = labels[:max_len-1]
            labels.append('[SEP]')
        # Convert tokens to IDs
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        # Create attention mask
        attention_mask = [1] * len(input_ids)
        tokenized_texts.append(tokens)
        word_piece_labels.append(labels)
    return tokenized_texts, word_piece_labels

In [11]:
# IMPROVEMENT 9: Custom BERT model with CRF layer for better sequence labeling
class BertCRF(nn.Module):
    def __init__(self, bert_model_name, num_labels):
        super(BertCRF, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        # Add a CRF layer
        self.crf = CRF(num_labels, batch_first=True)
    def forward(self, input_ids, token_type_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)
        if labels is not None:
            loss = -1 * self.crf(logits, labels, mask=attention_mask.byte())
            return loss, logits
        else:
            return logits

In [12]:
# Simple CRF implementation
class CRF(nn.Module):
    def __init__(self, num_tags, batch_first=False):
        super(CRF, self).__init__()
        self.num_tags = num_tags
        self.batch_first = batch_first
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
    def forward(self, emissions, tags, mask=None):
        # This is a simplified CRF implementation
        # In a real implementation, you would compute the full CRF loss
        score = torch.zeros(emissions.size(0)).to(emissions.device)
        for i in range(emissions.size(0)):
            for j in range(emissions.size(1) - 1):
                if mask is None or mask[i, j]:
                    score[i] += self.transitions[tags[i, j], tags[i, j+1]]
                    score[i] += emissions[i, j, tags[i, j]]
            score[i] += emissions[i, -1, tags[i, -1]]
        return score.mean()

In [13]:
# IMPROVEMENT 10: Better evaluation function with detailed metrics
def evaluate_model(model, dataloader, device, idx2tag):
    model.eval()
    predictions = []
    true_labels = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = tuple(t.to(device) for t in batch)
            input_ids, attention_mask, labels = batch
            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask)
            # Get predictions
            if isinstance(outputs, tuple):
                logits = outputs[1]
            else:
                logits = outputs
            # Get predictions from logits
            preds = torch.argmax(logits, dim=2)
            # Move to CPU
            preds = preds.detach().cpu().numpy()
            labels_np = labels.detach().cpu().numpy()
            attention_mask_np = attention_mask.detach().cpu().numpy()
            # Only keep predictions for valid tokens (exclude padding)
            for i, mask in enumerate(attention_mask_np):
                pred_i = []
                true_i = []
                for j, m in enumerate(mask):
                    if m:  # If token is not padding
                        pred_tag = idx2tag.get(preds[i][j], 'O')
                        true_tag = idx2tag.get(labels_np[i][j], 'O')
                        # Skip special tokens
                        if true_tag not in ['[CLS]', '[SEP]', 'X']:
                            pred_i.append(pred_tag)
                            true_i.append(true_tag)
                if pred_i:  # Only add if there are valid predictions
                    predictions.append(pred_i)
                    true_labels.append(true_i)
    # Calculate metrics
    print("\nEvaluation Results:")
    print("===================")
    # Overall metrics
    f1 = f1_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    print(f"Overall F1 Score: {f1:.4f}")
    print(f"Overall Precision: {precision:.4f}")
    print(f"Overall Recall: {recall:.4f}")
    # Detailed classification report
    print("\nDetailed Classification Report:")
    print(classification_report(true_labels, predictions))
    # IMPROVEMENT 11: Entity-specific metrics
    entity_metrics = {}
    # Extract unique entity types (excluding position markers B-, I-, etc.)
    entity_types = set()
    for tag in tag2idx.keys():
        if tag not in ['O', 'X', '[CLS]', '[SEP]']:
            entity_type = tag.split('-')[1] if '-' in tag else tag
            entity_types.add(entity_type)
    print("\nEntity-Specific Metrics:")
    print("=======================")
    for entity_type in sorted(entity_types):
        # Filter predictions and true labels for this entity type
        entity_preds = []
        entity_true = []
        for pred_seq, true_seq in zip(predictions, true_labels):
            entity_pred_seq = []
            entity_true_seq = []
            for p, t in zip(pred_seq, true_seq):
                # Check if prediction matches this entity type
                pred_entity = p.split('-')[1] if '-' in p and p != 'O' else ''
                true_entity = t.split('-')[1] if '-' in t and t != 'O' else ''
                # Mark as correct entity or O
                entity_pred_seq.append('B-ENT' if pred_entity == entity_type else 'O')
                entity_true_seq.append('B-ENT' if true_entity == entity_type else 'O')
            entity_preds.append(entity_pred_seq)
            entity_true.append(entity_true_seq)
        # Calculate metrics for this entity type
        try:
            entity_f1 = f1_score(entity_true, entity_preds)
            entity_precision = precision_score(entity_true, entity_preds)
            entity_recall = recall_score(entity_true, entity_preds)
            print(f"{entity_type}:")
            print(f"  F1: {entity_f1:.4f}")
            print(f"  Precision: {entity_precision:.4f}")
            print(f"  Recall: {entity_recall:.4f}")
            
            entity_metrics[entity_type] = {
                'f1': entity_f1,
                'precision': entity_precision,
                'recall': entity_recall
            }
        except Exception as e:
            print(f"{entity_type}: Error calculating metrics - {e}")
    return {
        'overall': {
            'f1': f1,
            'precision': precision,
            'recall': recall
        },
        'entity_metrics': entity_metrics
    }

In [16]:
# IMPROVEMENT 12: Main function with better error handling and logging
def main():
    try:
        # Set device
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Using device: {device}")
        # Load tokenizer
        tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)
        logger.info("Tokenizer loaded")
        # Load test data
        logger.info("Loading test data...")
        test_df = pd.read_json('./train.json')
        # Clean up dataframe
        if 'extras' in test_df.columns:
            test_df = test_df.drop(['extras'], axis=1)
        if 'metadata' in test_df.columns:
            test_df = test_df.drop(['metadata'], axis=1)
        # Extract entities
        logger.info("Extracting entities...")
        test_df['entities'] = get_entities(test_df)
        # Prepare test data
        logger.info("Preparing test data...")
        test_sentences, test_tags = get_test_data(test_df)
        logger.info(f"Processed {len(test_sentences)} test sentences")
        # Tokenize test data
        logger.info("Tokenizing test data...")
        tokenized_texts, word_piece_labels = get_tokenized_test_data(test_sentences, test_tags, tokenizer)
        # Convert to input format
        MAX_LEN = 256
        logger.info(f"Converting to model input format (max length: {MAX_LEN})...")
        input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
                                maxlen=MAX_LEN, dtype="long", truncating="post", padding="post")
        tags = pad_sequences([[tag2idx.get(l, tag2idx['O']) for l in lab] for lab in word_piece_labels], 
                            maxlen=MAX_LEN, value=tag2idx["O"], padding="post", 
                            dtype="long", truncating="post")
        attention_masks = [[float(i>0) for i in ii] for ii in input_ids]
        # Convert to PyTorch tensors
        test_inputs = torch.tensor(input_ids)
        test_tags = torch.tensor(tags)
        test_masks = torch.tensor(attention_masks)
        # Create DataLoader
        test_data = TensorDataset(test_inputs, test_masks, test_tags)
        test_sampler = SequentialSampler(test_data)
        test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=8)
        # Load model
        logger.info("Loading model...")
        try:
            # IMPROVEMENT 13: Try loading the enhanced model first
            model = BertCRF('bert-base-cased', len(tag2idx))
            model.load_state_dict(torch.load("fifth_epoch.pt", map_location=device))
            logger.info("Loaded enhanced BertCRF model")
        except Exception as e:
            logger.warning(f"Could not load enhanced model: {e}")
            logger.info("Falling back to standard BertForTokenClassification")
            # Load standard model
            model = BertForTokenClassification.from_pretrained(
                "bert-base-cased",
                num_labels=len(tag2idx)
            )
            try:
                # Try to load weights
                model.load_state_dict(torch.load("fifth_epoch.pt", map_location=device))
                logger.info("Loaded model weights successfully")
            except Exception as e:
                logger.warning(f"Error loading model weights: {e}")
                logger.info("Initializing with pre-trained BERT weights only")
        # Move model to device
        model.to(device)
        # Evaluate model
        logger.info("Evaluating model...")
        metrics = evaluate_model(model, test_dataloader, device, idx2tag)
        # Print summary
        logger.info("\nEvaluation Summary:")
        logger.info(f"Overall F1: {metrics['overall']['f1']:.4f}")
        logger.info(f"Overall Precision: {metrics['overall']['precision']:.4f}")
        logger.info(f"Overall Recall: {metrics['overall']['recall']:.4f}")
        # IMPROVEMENT 14: Save metrics to file
        import json
        with open('evaluation_metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
        logger.info("Metrics saved to evaluation_metrics.json")
    except Exception as e:
        logger.error(f"An error occurred: {e}")
        import traceback
        logger.error(traceback.format_exc())

In [17]:
if __name__ == "__main__":
    main()

2025-04-22 14:41:10,188 - INFO - Using device: cuda
2025-04-22 14:41:10,991 - INFO - loading vocabulary file https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-cased-vocab.txt from cache at C:\Users\user\.pytorch_pretrained_bert\5e8a2b4893d13790ed4150ca1906be5f7a03d6c4ddf62296c383f6db42814db2.e13dbb970cb325137104fb2e5f36fe865f27746c6b526f6352861b1980eb80b1
2025-04-22 14:41:11,030 - INFO - Tokenizer loaded
2025-04-22 14:41:11,031 - INFO - Loading test data...
2025-04-22 14:41:11,544 - INFO - Extracting entities...
2025-04-22 14:41:11,545 - WARNING - Unknown label: College
2025-04-22 14:41:11,546 - WARNING - Unknown label: College
2025-04-22 14:41:11,546 - WARNING - Unknown label: Can Relocate to
2025-04-22 14:41:11,547 - WARNING - Unknown label: University
2025-04-22 14:41:11,547 - WARNING - Unknown label: Links
2025-04-22 14:41:11,548 - WARNING - Unknown label: Links
2025-04-22 14:41:11,548 - WARNING - Unknown label: Links
2025-04-22 14:41:11,549 - WARNING - Unknown label: L


Evaluation Results:
Overall F1 Score: 0.0024
Overall Precision: 0.0012
Overall Recall: 0.0215

Detailed Classification Report:


C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         CLG       0.00      0.32      0.00        19
        CLS]       0.00      0.00      0.00         0
     COMPANY       0.03      0.09      0.04       225
         DEG       0.00      0.00      0.00        19
       DESIG       0.00      0.00      0.00       333
       EMAIL       0.00      0.00      0.00       144
    GRADYEAR       0.00      0.00      0.00        21
         LOC       0.00      0.00      0.00       173
        NAME       0.00      0.00      0.00       478
        SEP]       0.00      0.00      0.00         0
      SKILLS       0.00      0.00      0.00       179
         YOE       0.01      0.21      0.02        39
           _       0.00      0.00      0.00         0

   micro avg       0.00      0.02      0.00      1630
   macro avg       0.00      0.05      0.00      1630
weighted avg       0.00      0.02      0.01      1630


Entity-Specific Metrics:
CLG:
  F1: 0.0045
  Precision: 0.0023
  Recall: 0.776

2025-04-22 14:42:46,492 - INFO - 
Evaluation Summary:
2025-04-22 14:42:46,492 - INFO - Overall F1: 0.0024
2025-04-22 14:42:46,493 - INFO - Overall Precision: 0.0012
2025-04-22 14:42:46,493 - INFO - Overall Recall: 0.0215
2025-04-22 14:42:46,495 - INFO - Metrics saved to evaluation_metrics.json


SKILLS:
  F1: 0.0044
  Precision: 0.0082
  Recall: 0.0030
YOE:
  F1: 0.0830
  Precision: 0.0472
  Recall: 0.3451
